# ML-08 ? Capstone Modeling Lane

## 1. Method choice and why

I use a logistic regression as the readable reference and a random forest as the non-linear comparison. Both are classifiers only so they can produce a probability-like ranking score; the decision remains ?which pages should a reviewer inspect first?? The outcome is a March proxy: later-half impressions are below 80% of earlier-half impressions. It is a measured forward window within this development month, not evidence that a refresh will cause recovery.

In [2]:
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy
import os, duckdb, numpy as np, pandas as pd, json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, average_precision_score
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets or the HF_TOKEN environment variable first."
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
pages = con.sql(f"""
SELECT client_hash_id, content_hash_id,
 SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impressions_prev,
 SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks ELSE 0 END) AS clicks_prev,
 AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_avg_position END) AS avg_position_prev,
 STDDEV(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_avg_position END) AS position_volatility_prev,
 COUNT(CASE WHEN report_date < DATE '2026-03-16' AND gsc_impressions > 0 THEN 1 END) AS active_days_prev,
 SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impressions_outcome
FROM {FACT} GROUP BY 1,2 HAVING impressions_prev >= 100
""").df()
pages["is_declining"] = (pages.impressions_outcome < 0.8 * pages.impressions_prev).astype(int)
features=["impressions_prev","clicks_prev","avg_position_prev","position_volatility_prev","active_days_prev"]
pages["log_impressions_prev"]=np.log1p(pages.impressions_prev); pages["log_clicks_prev"]=np.log1p(pages.clicks_prev)
features=["log_impressions_prev","log_clicks_prev","avg_position_prev","position_volatility_prev","active_days_prev"]
def p_at_k(y,s,k):
 k=min(k,len(y)); return float(pd.DataFrame({"y":np.asarray(y),"s":np.asarray(s)}).nlargest(k,"s").y.mean())
def baseline_score(d):
 return d.impressions_prev * (1 + (d.position_volatility_prev.fillna(0) >= 5).astype(int))
print(f"Page-level March frame: {len(pages):,} rows; positive rate: {pages.is_declining.mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level March frame: 77,540 rows; positive rate: 28.5%


## 2. Split design

The split holds out entire `client_hash_id` groups (25%, seed 42). That is stricter than a row split because no other page from a held-out client can teach the model its client-specific traffic pattern. Features use March 1?15; the outcome uses March 16?31. `impressions_outcome` exists only to create the label and is never in `features`.

In [3]:
# The grouped split and group-disjoint assertion execute in the next cell.


## 3. Train + compare vs my baseline

The Week 4 rule ranks prior impressions, doubled when prior position volatility is at least five. The same held-out pages, labels, and Precision@K evaluate every method. Average precision is included as whole-ranking context; Precision@20/50 remains the decision metric.

In [4]:
splitter=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
tr,te=next(splitter.split(pages[features],pages.is_declining,groups=pages.client_hash_id))
train,test=pages.iloc[tr].copy(),pages.iloc[te].copy()
assert not set(train.client_hash_id).intersection(test.client_hash_id)
models={
 "logistic_regression":Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42))]),
 "random_forest":Pipeline([("impute",SimpleImputer(strategy="median")),("model",RandomForestClassifier(n_estimators=300,min_samples_leaf=25,class_weight="balanced_subsample",random_state=42,n_jobs=-1))])}
rows=[]; fitted={}
for name,model in models.items():
 model.fit(train[features],train.is_declining); score=model.predict_proba(test[features])[:,1]; fitted[name]=model
 rows.append({"method":name,"precision@20":p_at_k(test.is_declining,score,20),"precision@50":p_at_k(test.is_declining,score,50),"average_precision":average_precision_score(test.is_declining,score)})
bscore=baseline_score(test)
rows.insert(0,{"method":"week_4_baseline","precision@20":p_at_k(test.is_declining,bscore,20),"precision@50":p_at_k(test.is_declining,bscore,50),"average_precision":average_precision_score(test.is_declining,bscore)})
comparison=pd.DataFrame(rows).sort_values("precision@50",ascending=False).reset_index(drop=True)
comparison
output_dir=Path("work/outputs"); output_dir.mkdir(parents=True,exist_ok=True)
metrics={"split":"GroupShuffleSplit by client_hash_id, seed 42","rows":int(len(pages)),"test_rows":int(len(test)),"positive_rate":float(test.is_declining.mean()),"results":comparison.to_dict(orient="records")}
(output_dir/"w05_model_metrics.json").write_text(json.dumps(metrics,indent=2))
print("Held-out positive rate:",f"{test.is_declining.mean():.1%}")
print("Saved safe metric receipt: work/outputs/w05_model_metrics.json")
comparison


Held-out positive rate: 14.7%
Saved safe metric receipt: work/outputs/w05_model_metrics.json


,method,precision@20,precision@50,average_precision
0,logistic_regression,0.20,0.32,0.192374
1,week_4_baseline,0.10,0.16,0.131939
2,random_forest,0.15,0.14,0.162502


## 4. Errors and interpretation

Feature importance is descriptive, not causal. I inspect the random forest?s held-out false positives and false negatives without exposing any raw client data. A mistaken pick can reflect temporary movement, data noise, or signals that this five-feature frame does not observe.

In [5]:
best=fitted["random_forest"]; imp=pd.DataFrame({"feature":features,"importance":best.named_steps["model"].feature_importances_}).sort_values("importance",ascending=False)
test["model_score"]=best.predict_proba(test[features])[:,1]; test["predicted"]=test.model_score>=.5
errors=test.loc[test.predicted != test.is_declining,["content_hash_id","is_declining","model_score","impressions_prev","avg_position_prev","position_volatility_prev"]].head(3)
print("Top descriptive feature importances:"); display(imp)
print("Three held-out wrong cases (pseudonymous IDs only):"); display(errors)
print("Interpretation: importance ranks how this fitted model used observed signals; it does not show that changing a signal changes decline.")


Top descriptive feature importances:


,feature,importance
2,avg_position_prev,0.274081
3,position_volatility_prev,0.241304
0,log_impressions_prev,0.226402
1,log_clicks_prev,0.149063
4,active_days_prev,0.109151


Three held-out wrong cases (pseudonymous IDs only):


,content_hash_id,is_declining,model_score,impressions_prev,avg_position_prev,position_volatility_prev
1986,content_b2c6482a36345436,0,0.588986,1628.0,2.927786,1.572509
1993,content_200a5e98bfb089f7,0,0.532164,228.0,37.802629,9.200738
1996,content_b8fdb09fec9dc62e,0,0.564962,1958.0,6.940152,1.720616


Interpretation: importance ranks how this fitted model used observed signals; it does not show that changing a signal changes decline.


## Self-check

- [x] Grouped, out-of-sample comparison; no label-derived feature
- [x] Precision@K and the held-out base rate are printed
- [x] Results are computed live; no figures or metrics are pre-written
- [x] IDs are pseudonymous and used only for grouping or review